# 15.1 Why Test, and the `assert` Statement

**Prerequisites:** 06 Exception Handling (especially 6.1), 04 Functions, 07 Module and Packages  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Why automated tests exist, and what they actually buy you
- `assert` — anatomy, and what it is *for*
- 🔴 `python -O` deletes every `assert` — measured on the bytecode
- Why validation must raise, not assert
- **Arrange – Act – Assert**, the shape of every test you will ever write
- Building a working test runner in 20 lines, so `pytest` is not magic
- Unit, integration and end-to-end — and the test pyramid
- 🔴 What makes a test *bad*: order dependence, demonstrated

---

## The problem tests solve

You wrote `retry_delay()` in March. In August someone changes it to add jitter. Does it still
back off correctly? Does it still cap at the ceiling? Does it still return `0` for the first
attempt?

You have three options:

| Approach | Cost per check | Cost after 50 changes |
|---|---|---|
| Read the code and reason | minutes, and you can be wrong | you stop bothering |
| Run it by hand and eyeball the output | ~30 seconds | ~25 minutes, and you skip cases |
| Run an automated test | ~0.01 seconds | ~0.5 seconds, and it never skips |

A test is **an executable claim about your code**. Not documentation that goes stale, not a
comment that lies — a claim the machine re-checks every time.

> The real payoff is not "finding bugs". It is **being able to change code without fear**.
> Untested code calcifies: nobody dares touch it, so it rots.

This folder builds up in the order the tools were invented: `assert` (here), then `unittest`
(**15.2**), then `pytest` (**15.3**), then fixtures, mocking and the rest.

## Where you are starting from

You have already been testing — by printing and reading.

Below is a retry policy with exponential backoff and a ceiling. The checking is done the way
almost everyone does it before they meet a test framework.

In [ ]:
def retry_delay(attempt, base=1.0, ceiling=30.0):
    """Seconds to wait before retry number `attempt` (0-based)."""
    return min(base * 2 ** attempt, ceiling)


# "Testing" by printing and reading:
for attempt in range(8):
    print(f"  attempt {attempt}: {retry_delay(attempt):5.1f}s")

Look at what just happened. **The computer printed; you did the checking.**

That is the whole problem:

- You have to *read* eight lines and decide whether each is right.
- You have to remember what "right" was.
- Nothing fails. If `retry_delay` starts returning `-1`, the cell still runs happily and prints
  `-1.0` — you only notice if you are paying attention.
- Tomorrow you will not re-read it.

The fix is to move the checking **into the code**, so the machine does it.

## `assert` — the smallest possible test

```
assert  retry_delay(0) == 1.0  ,  "first retry should be immediate-ish"
──┬───  ─────────┬─────────────    ─────────────┬──────────────────────
  │              │                              │
  │              │                              └─ optional message, shown on failure
  │              └─ any expression; truthy = pass, falsy = raise
  └─ statement, not a function. `assert(x, "msg")` is a BUG — see below.
```

`assert EXPR, MSG` is exactly equivalent to:

```python
if not EXPR:
    raise AssertionError(MSG)
```

You met `assert` in **6.1**. Here it becomes the foundation of everything else — `pytest`'s
entire assertion system is built on this one statement.

In [ ]:
# The same eight checks, but now the machine does the checking.
assert retry_delay(0) == 1.0,  "attempt 0 should wait 1s"
assert retry_delay(1) == 2.0,  "delays should double"
assert retry_delay(2) == 4.0,  "delays should double"
assert retry_delay(5) == 30.0, "should be capped at the ceiling"
assert retry_delay(9) == 30.0, "should stay capped, however high the attempt"

print("all 5 checks passed - and nothing needed reading")

# Now watch one fail, so you can see what failure looks like.
try:
    assert retry_delay(3) == 9.0, "delays should double"
except AssertionError as exc:
    print(f"\nAssertionError: {exc}")
    print("  (retry_delay(3) is actually", retry_delay(3), "- the *test* was wrong here)")

### A helper for the rest of this notebook

Several things below can only be shown by running a **separate interpreter** — a different
command-line flag, or a compile-time warning that this notebook's own settings would suppress.
So here is a small helper that writes a script to a temporary directory and runs it.

This is the same pattern used in **12.3**, and it keeps the notebook re-runnable: nothing is
written into the repository, and the scratch directory is deleted at the end.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
import shutil
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py151_"))


def run_script(name, source, *flags):
    """Write a script, run it in a fresh interpreter, return combined output."""
    path = WORK / name
    path.write_text(textwrap.dedent(source), encoding="utf-8")
    done = subprocess.run([sys.executable, *flags, str(path)],
                          capture_output=True, text=True, timeout=120)
    return (done.stdout + done.stderr).strip()


print("scratch:", WORK)

### 🔴 The trap: `assert` with parentheses

`assert` is a **statement**, not a function. Writing it like a function call silently breaks
it: `assert (expr, msg)` asserts a **2-tuple**, and a non-empty tuple is always truthy — so
the check can never fail.

In [ ]:
print(run_script("tuple_assert.py", """
    def retry_delay(attempt, base=1.0, ceiling=30.0):
        return min(base * 2 ** attempt, ceiling)

    wrong = (retry_delay(3) == 999.0, "this message is never shown")
    print("  the tuple being asserted:", wrong)
    print("  bool(that tuple)        :", bool(wrong), " <- truthy, so assert always passes")

    assert (retry_delay(3) == 999.0, "never fires")      # passes! the claim is false!
    print("  ^ that assert 'passed' while claiming 8.0 == 999.0")

    try:
        assert retry_delay(3) == 999.0, "this one really fires"
    except AssertionError as exc:
        print("  correct form (no parentheses) raises:", exc)
"""))

Note the **`SyntaxWarning: assertion is always true, perhaps remove parentheses?`** in that
output. Python does try to warn you — but a warning is easy to miss in a busy test run, and
it is only emitted when the file is *compiled*, so a cached `.pyc` will not show it twice.

🔴 Treat it as an error and it cannot slip past you:

```bash
python -W error::SyntaxWarning -m pytest
```

## 🔴 `assert` is for programmer errors — never for validation

This is the single most important thing in this notebook, and it is not obvious.

**`python -O` removes every `assert` statement from your program.** Not "skips" — *removes*,
at compile time. The bytecode does not contain them.

So any check you need in production **must not be an `assert`**.

The next cell proves it by running the same file twice, with and without `-O`.

In [ ]:
DISCOUNT = """
    def apply_discount(total, percent):
        assert 0 <= percent <= 100, "percent out of range"
        return total * (1 - percent / 100)

    print("  apply_discount(200.0, 150) =", apply_discount(200.0, 150))
"""

print("--- normal:  python discount.py ---")
print(run_script("discount.py", DISCOUNT))

print("\n--- optimised:  python -O discount.py ---")
print(run_script("discount.py", DISCOUNT, "-O"))

Read that again. With `-O`, a **150% discount** sailed through and produced a
**negative price**. The guard did not "fail to trigger" — it was not in the program at all.

Here is the same thing at the bytecode level, which is the honest proof.

In [ ]:
BYTECODE = """
    import dis

    def apply_discount(total, percent):
        assert 0 <= percent <= 100, "percent out of range"
        return total * (1 - percent / 100)

    opnames = [i.opname for i in dis.get_instructions(apply_discount)]
    print("__debug__            :", __debug__)
    print("bytecode instructions:", len(opnames))
    print("has RAISE_VARARGS    :", "RAISE_VARARGS" in opnames)
    print("message in co_consts :",
          "percent out of range" in apply_discount.__code__.co_consts)
"""

print("--- normal ---")
print(run_script("bytecode.py", BYTECODE))
print("\n--- python -O ---")
print(run_script("bytecode.py", BYTECODE, "-O"))

| | normal | `python -O` |
|---|---|---|
| `__debug__` | `True` | `False` |
| instructions in the function | **29** | **9** |
| `RAISE_VARARGS` opcode present | yes | **no** |
| `"percent out of range"` in `co_consts` | yes | **no** |

Twenty of the twenty-nine instructions are gone, and **the failure message is not even stored
in the compiled function any more**. There is nothing left to trigger. So:

| Use | Right tool | Why |
|---|---|---|
| "this input from a user/network/file is bad" | `raise ValueError(...)` | must survive `-O`; it is an *expected* condition |
| "my own code has a bug — this can't happen" | `assert` | a claim about the program, not the data |
| "this test's expectation is not met" | `assert` | test runners never run with `-O` |

Rewritten correctly — validation raises, the internal invariant asserts:

```python
def apply_discount(total, percent):
    if not 0 <= percent <= 100:                     # survives -O
        raise ValueError(f"percent must be 0-100, got {percent}")
    result = total * (1 - percent / 100)
    assert result <= total, "discount increased the price"   # my-bug check
    return result
```

See **6.2** for designing the exception types you raise here.

## Arrange – Act – Assert

Every test, in every framework, in every language, has the same three parts. Writing them in
this order — and putting a blank line between them — makes tests readable at a glance.

```
def test_delay_is_capped_at_the_ceiling():
    attempt, ceiling = 9, 30.0          ARRANGE   set up the inputs
                                                  and the world the code needs
    delay = retry_delay(attempt,
                        ceiling=ceiling) ACT      call the thing under test,
                                                  exactly once
    assert delay == ceiling              ASSERT   state what should be true
```

Three rules that follow from the shape:

1. **One Act per test.** If you call the function three times, you have three tests.
2. **Assert on the outcome, not the mechanism.** Check *what* it returned, not *how* it got there.
3. **The test name is the claim.** `test_delay_is_capped_at_the_ceiling` tells you what broke
   without reading the body.

In [ ]:
def test_first_attempt_waits_the_base_delay():
    base = 1.0                                    # arrange

    delay = retry_delay(0, base=base)             # act

    assert delay == base                          # assert


def test_delay_doubles_each_attempt():
    delays = [retry_delay(n, base=1.0) for n in range(4)]

    assert delays == [1.0, 2.0, 4.0, 8.0]


def test_delay_is_capped_at_the_ceiling():
    ceiling = 30.0

    delay = retry_delay(9, base=1.0, ceiling=ceiling)

    assert delay == ceiling


def test_ceiling_applies_even_when_base_exceeds_it():
    delay = retry_delay(0, base=100.0, ceiling=30.0)

    assert delay == 30.0


for test in (test_first_attempt_waits_the_base_delay,
             test_delay_doubles_each_attempt,
             test_delay_is_capped_at_the_ceiling,
             test_ceiling_applies_even_when_base_exceeds_it):
    test()
print("4 tests, called by hand, all passed")

## You have already written a test framework

In **14.16** you built `verify(fast, slow, generate)` — run two implementations on random
inputs and compare. That is a real testing technique (property-based testing; it comes back
properly in **15.6**).

The loop at the end of the last cell is the other half: **find the tests, run them, report**.
Six lines short of a test runner. Let's finish it — because once you have written one,
`pytest` stops being magic and becomes "the one I didn't have to write".

In [ ]:
import traceback


def run_tests(namespace, prefix="test_"):
    """A working test runner. This is genuinely what the frameworks do."""
    tests = [(name, obj) for name, obj in sorted(namespace.items())
             if name.startswith(prefix) and callable(obj)]

    passed, failed = [], []
    for name, test in tests:
        try:
            test()
        except AssertionError as exc:
            failed.append((name, "FAIL", exc, traceback.format_exc()))
        except Exception as exc:                      # noqa: BLE001 - runner must catch all
            failed.append((name, "ERROR", exc, traceback.format_exc()))
        else:
            passed.append(name)

    for name, kind, exc, _tb in failed:
        detail = f": {exc}" if str(exc) else ""
        print(f"  {kind:5} {name}{detail}")
    print(f"\n  {len(passed)} passed, {len(failed)} failed, "
          f"{len(tests)} collected")
    return not failed


run_tests(globals())

Now give it something broken, so the report has to earn its keep. `retry_delay_buggy`
forgets the ceiling and mis-handles attempt 0.

In [ ]:
def retry_delay_buggy(attempt, base=1.0, ceiling=30.0):
    return base * 2 ** attempt          # 🔴 no ceiling applied


def check_first_attempt():
    assert retry_delay_buggy(0) == 1.0


def check_doubling():
    assert [retry_delay_buggy(n) for n in range(4)] == [1.0, 2.0, 4.0, 8.0]


def check_ceiling():
    assert retry_delay_buggy(9) == 30.0, f"got {retry_delay_buggy(9)}, want 30.0"


def check_negative_attempt_rejected():
    retry_delay_buggy(-1)["boom"]       # a genuine error, not an assertion failure


run_tests(globals(), prefix="check_")

Two things that report shows, and both matter:

- **`FAIL` vs `ERROR` are different.** A `FAIL` is "the code ran and gave the wrong answer".
  An `ERROR` is "the code blew up before we could even check". `unittest` keeps this
  distinction (**15.2**); `pytest` deliberately drops it.
- **Every test ran.** A failing test does not stop the others. That is why the runner catches
  rather than lets exceptions propagate.

### What our 20 lines are missing

| Missing | Why you want it | Covered in |
|---|---|---|
| Finding tests across *files* | you will have hundreds | **15.3** |
| Showing *why* an assert failed (the values) | `assert a == b` tells you nothing | **15.3** |
| Setup/teardown shared between tests | temp dirs, fake clocks, DB rows | **15.4** |
| Running one test, or a subset | 900 tests, one is broken | **15.3** |
| Replacing real dependencies | don't hit the network in a unit test | **15.5** |
| Skips, expected failures, marks | not every test applies everywhere | **15.2**, **15.3** |

That is the entire rest of this folder.

## Unit, integration, end-to-end

The words get used loosely. The distinction that matters in practice is **how much of the
system a failure implicates**.

```
        ▲            ┌─────────────────┐   E2E / system
   few  │            │   e2e (5-20)    │   real browser, real DB, real network
        │            └─────────────────┘   slow (seconds-minutes), flaky, priceless
        │        ┌─────────────────────────┐
        │        │   integration (50-200)  │   your code + one real dependency
        │        └─────────────────────────┘   (a real sqlite file, a real HTTP server)
        │   ┌───────────────────────────────────┐
  many  │   │        unit  (500-5000)           │   one function/class, nothing external
        ▼   └───────────────────────────────────┘   microseconds, deterministic
```

| | Unit | Integration | End-to-end |
|---|---|---|---|
| Scope | one function or class | two or three components | the whole system |
| Speed | µs | ms | seconds+ |
| When it fails you know | exactly what broke | roughly where | only *that* something broke |
| Should be | the vast majority | some | a handful |

🔴 **The pyramid is a shape, not a law.** Its real message: *the further a test reaches, the
more it costs and the less precisely it points*. Prefer the cheapest test that would actually
catch the bug you are worried about.

You have already written integration tests without the name — **10.3** against a real SQLite
file, **11.2** against a real socket.

## 🔴 What makes a test bad

A test that passes when the code is broken, or fails when it isn't, is worse than no test —
it burns trust. The most common cause is a test that depends on something it does not control.

The next cell has two tests that are individually fine and **give different results depending
on the order they run in**, because they share mutable state. This is not hypothetical; it is
the most common flake in real suites.

In [ ]:
# A module-level cache -- shared, mutable, and nobody's property.
SESSION_CACHE = {}


def cache_session(user_id, token):
    SESSION_CACHE[user_id] = token
    return len(SESSION_CACHE)


def check_cache_starts_empty():
    assert len(SESSION_CACHE) == 0, f"expected empty, found {len(SESSION_CACHE)}"


def check_caching_adds_an_entry():
    assert cache_session(7, "abc") == 1


order_a = ["check_cache_starts_empty", "check_caching_adds_an_entry"]
order_b = ["check_caching_adds_an_entry", "check_cache_starts_empty"]

for label, order in (("A", order_a), ("B", order_b)):
    SESSION_CACHE.clear()
    results = []
    for name in order:
        try:
            globals()[name]()
            results.append(f"{name} PASS")
        except AssertionError as exc:
            results.append(f"{name} FAIL ({exc})")
    print(f"  order {label}: " + "  |  ".join(results))

SESSION_CACHE.clear()
print("\nSame two tests. Same code. Different verdict. The tests are the bug.")

Order **A** passes; order **B** fails — and `pytest -p randomly` or simply
renaming a test can flip you from one to the other.

The cure is **isolation**: every test creates the state it needs and leaves nothing behind.
That is what fixtures are for (**15.4**).

### The four properties worth remembering

| Property | Means | Broken by |
|---|---|---|
| **Fast** | milliseconds, so you run them constantly | real network, `time.sleep`, big fixtures |
| **Isolated** | any order, any subset, same result | shared globals, leftover files, DB rows |
| **Repeatable** | same result on any machine, any day | `datetime.now()`, `random` without a seed, locale, timezone |
| **Self-validating** | pass/fail, no human reading output | tests that `print` instead of `assert` |

## Naming and layout

```
project/
├── src/orders/
│   ├── __init__.py
│   └── retry.py
└── tests/
    ├── test_retry.py            # mirrors src/orders/retry.py
    └── test_orders_api.py
```

- Test **files**: `test_*.py` (this is what the runners look for by default — **15.3**).
- Test **functions**: `test_<what>_<expected behaviour>`.
  `test_delay_is_capped_at_the_ceiling`, not `test_2` or `test_retry_delay`.
- One assertion *concept* per test. Several `assert` lines checking one claim is fine;
  three unrelated claims is three tests.

> Read the failure line of a good suite and you should not need to open the file:
> `FAILED tests/test_retry.py::test_delay_is_capped_at_the_ceiling`. That is the bug report.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **`assert (expr, msg)`** — asserting a tuple, which is always truthy, so the check can never fail. Never put parentheses around the pair.
2. 🔴 **Using `assert` to validate input.** `python -O` deletes it and your validation silently disappears. Raise `ValueError`/`TypeError` instead — see **6.2**.
3. **Testing by printing.** If a human has to read the output to know whether it passed, it is not a test.
4. **Tests that share mutable state** — module-level dicts, files in the working directory, database rows. They pass alone and fail in a suite, or vice versa.
5. **Depending on the clock, the network, the locale or unseeded `random`.** The test now asserts something about the world, not about your code.
6. **Asserting on the mechanism instead of the outcome** — checking that a helper was called rather than that the answer is right. Such tests break on every refactor.
7. **One giant `test_everything`.** When it fails you learn nothing except that something is wrong; and it stops at the first failure, hiding the rest.
8. **Writing tests only for the happy path.** The bugs live in the empty list, the negative number, the duplicate key and the value exactly on the boundary.

## Best Practices

- Write the test name as the claim: `test_<subject>_<expected behaviour>`.
- Keep Arrange / Act / Assert visually separated by blank lines — one Act per test.
- Prefer the cheapest test that would catch the bug: unit over integration over e2e.
- Make every test create what it needs and clean up after itself (**15.4** automates this).
- Test the boundaries: 0, 1, empty, exactly-at-the-limit, one-past-the-limit.
- When you fix a bug, first write the test that fails because of it. Then fix it.
- Use `assert` freely *inside tests*; use explicit `raise` for anything production relies on.
- Run the suite before you commit, not before you release.

## Practice Exercises

Try these before moving on.

1. Write four tests for `retry_delay` covering: attempt 0, doubling, the ceiling, and a `base` already larger than the ceiling. Run them with `run_tests`.
2. Add jitter to `retry_delay` (`delay * random.uniform(0.8, 1.0)`) and try to keep your tests passing. What has to change? (Hint: the answer is *not* a fixed seed — see **15.5**.)
3. 🔴 Take the `SESSION_CACHE` example and make the two tests pass in **both** orders without deleting either test.
4. Put `assert (1 == 2, 'boom')` in a file and confirm it passes while emitting a `SyntaxWarning`. Then re-run with `python -W error::SyntaxWarning` and confirm it becomes fatal. Finally, write a check that finds this mistake anywhere in a file (hint: `ast`, from **7.3** — look for an `ast.Assert` whose `test` is an `ast.Tuple`).
5. Take any function you wrote in **04 Functions** and add `assert` checks for its invariants. Then run the file with `python -O` and confirm they vanish.
6. Classify these as unit, integration or e2e: (a) `test_parse_iso_date`, (b) a test that writes to a real SQLite file, (c) a test that starts your web app and logs in. Which would you want most of?

---

## Version notes

| Version | Change |
|---|---|
| 3.12 | `unittest`'s deprecated aliases (`assertEquals`, `failUnless`, …) **removed** — see **15.2** |
| 3.11 | `assert` failure tracebacks gained fine-grained error locations (the `^^^^` markers under the exact sub-expression) |
| 3.9+ | `python -O` behaviour here is unchanged and long-standing — `assert` has always been strippable |

## Where next

| Notebook | Covers |
|---|---|
| **15.2 unittest** | the standard library's framework — `TestCase`, lifecycle, discovery |
| **15.3 pytest** | plain `assert` with real failure output, parametrisation, the CLI |
| **15.4 Fixtures** | isolation and setup done properly |
| **15.5 Test Doubles** | replacing the network, the clock and the database |
| **15.6 In Practice** | coverage, property-based testing, CI |

## Related

- **6.1 Exception Handling** — where `assert` was introduced
- **6.2 Custom Exceptions** — designing the exceptions your validation raises
- **14.16 Interview Patterns** — the `verify()` harness this notebook picked up
- **7.2 Python Packages** — the `src/` + `tests/` layout